# Under the Clouds: train the flood U-Net (Colab / Kaggle)

Trains a U-Net (ResNet-34) to find **water** in Sentinel-1 radar chips from the Sen1Floods11 dataset, then compares it with simple threshold baselines on the held-out **test** scenes.

**Before you start**
1. Turn on a GPU: *Runtime > Change runtime type > T4 GPU* (Kaggle: *Settings > Accelerator > GPU*, and turn **Internet** on).
2. The project code must be on GitHub (push it first) or uploaded as a zip into this session.
3. Run the cells from top to bottom. The dataset download is about 650 MB.

**Honest note:** this notebook has not been run on a GPU yet. The code was tested on the CPU with a few real chips. Write down whatever numbers you actually get; do not copy numbers from anywhere else.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: training will be very slow. Change the runtime type first.")

## 1. Get the code

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/Rabaaab/under-the-clouds.git"   # change if your repo lives elsewhere
REPO_DIR = "under-the-clouds"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

assert os.path.exists("src/utc/train.py"), (
    "The code is not in the repo yet. Push it to GitHub, or upload the project folder as a zip and unzip it here."
)
print("Working in", os.getcwd())

## 2. Install what Colab does not have (PyTorch is already installed)

In [ ]:
!pip install -q segmentation-models-pytorch rasterio geopandas shapely
import segmentation_models_pytorch as smp, rasterio
print("smp", smp.__version__, "| rasterio", rasterio.__version__)

## 3. Download Sen1Floods11 (hand-labelled subset, about 650 MB)

In [ ]:
!python scripts/sen1floods11.py download --root data/sen1floods11

## 4. Cut the scenes into training chips (one folder per split, so test data never leaks into training)

In [ ]:
!python scripts/sen1floods11.py prepare --root data/sen1floods11 --out data/chips

## 5. Train

ImageNet-pretrained ResNet-34 encoder, BCE + Dice loss, Adam, flips and rotations as augmentation. The best checkpoint is chosen by **validation IoU**. If the pretrained weights fail to download, add `--no-pretrained` (results will be worse).

Try a few settings (for example `--lr 3e-4`, `--epochs 40`) and keep notes of what each run gave.

In [ ]:
!python scripts/train.py --train-dir data/chips/train --val-dir data/chips/valid --epochs 30 --batch-size 16 --lr 1e-4 --out outputs/unet.pt

## 6. Evaluate on the held-out test scenes

Scene-level IoU and F1 pooled over valid pixels. Compared methods:
- `baseline_vv`, `baseline_vh`: Otsu threshold per scene (the naive approach)
- `tuned_vv`, `tuned_vh`: one global threshold fitted on the training scenes (the **strongest simple baseline**)
- `cnn`: the trained U-Net

Run the test split **once**, after you have chosen your final settings.

In [ ]:
!python scripts/sen1floods11.py evaluate --root data/sen1floods11 --split valid --checkpoint outputs/unet.pt --device cuda --out outputs/eval_valid.json
!python scripts/sen1floods11.py evaluate --root data/sen1floods11 --split test  --checkpoint outputs/unet.pt --device cuda --out outputs/eval_test.json

In [ ]:
import json, pandas as pd

def table(path):
    d = json.load(open(path))
    rows = {k: v for k, v in d.items() if isinstance(v, dict) and "iou" in v}
    return pd.DataFrame(rows).T[["iou", "f1", "precision", "recall"]].round(3), d

for name in ("valid", "test"):
    t, d = table(f"outputs/eval_{name}.json")
    print(f"=== {name} ({d['n_scenes']} scenes) ===")
    display(t)
print("Tuned thresholds (dB):", d.get("tuned_thresholds_db"))

## 7. Save the results

Download the checkpoint and the two JSON files and give them to the team (do **not** commit the `.pt` file to GitHub).
Write the **test** numbers for `tuned_vh` / `tuned_vv` and `cnn` in the README results table, exactly as printed.

In [ ]:
try:
    from google.colab import files
    for f in ("outputs/unet.pt", "outputs/eval_valid.json", "outputs/eval_test.json"):
        files.download(f)
except ImportError:
    print("Not on Colab: find the files in the 'outputs' folder (on Kaggle they are in /kaggle/working).")